In [3]:
# dependencies
!pip install transformers datasets evaluate accelerate -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.1 MB/s eta 0:00:00


In [4]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer,AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import evaluate


In [5]:
device = ('cuda' if torch.cuda.is_available()
          else 'mps' if torch.backends.mps.is_available()
          else 'cpu')
device

'cuda'

# load datase

In [7]:
ag_news_ds = load_dataset('ag_news')


In [8]:
print (ag_news_ds)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [9]:
#first sample
ag_news_ds['train'][0]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

In [22]:
label_names = ag_news_ds["train"].features["label"].names

# Tokenization

In [13]:
# model , non_sensetive to lower/captil letters
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(ex):
  return tokenizer(
      ex['text'],
      truncation= True,
      padding= False,
      max_length= 128,

  )


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [15]:
tokenize_dataset = ag_news_ds.map(
    tokenize_function,
    batched = True,
    remove_columns = ['text'] #just tokenz remain
)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [19]:
print(tokenize_dataset["train"][0])

{'label': 2, 'input_ids': [101, 2813, 2358, 1012, 6468, 15020, 2067, 2046, 1996, 2304, 1006, 26665, 1007, 26665, 1011, 2460, 1011, 19041, 1010, 2813, 2395, 1005, 1055, 1040, 11101, 2989, 1032, 2316, 1997, 11087, 1011, 22330, 8713, 2015, 1010, 2024, 3773, 2665, 2153, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [20]:
#padding *
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [23]:
# Model loading and Label Mapping

id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}
# {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}

In [25]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    #class count
    num_labels=4,
    # for inference: class name > num
    id2label=id2label,
    # for training: num > name
    label2id=label2id,
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [26]:
#model architect
print(model)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [31]:
# num of parameters
total_param = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(total_param)

# All fine-tunes
print(trainable)

66956548
66956548


In [42]:
# Metrics, evaluate
accu_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metric(eval_pred):
  logits, labels = eval_pred

  #highest logit showen class of predict
  predict = np.argmax(logits, axis=-1)

  acc = accu_metric.compute(
      predictions = predict,
      references = labels,
  )

  f1 = f1_metric.compute(
      predictions = predict,
      references = labels,
      average="weighted" # suit to multy classes

  )

  return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

# Training arguments

In [36]:
train_args = TrainingArguments(

    output_dir="./distilbert-ag-news",

    # num of batches & epochs
    num_train_epochs=3,
    # num of Epoches
    per_device_train_batch_size=32,
    # no need big gradient in eval
    per_device_eval_batch_size=64,

    # Learning Rate
    learning_rate = 2e-5, # this mount standard for fine-tuning bert
    # L2 regularization
    weight_decay=0.01,

    warmup_ratio=0.1,

    # evaluation

    # evaluate after every epoch
    eval_strategy="epoch",
    #save after every epoch
    save_strategy="epoch",
    # load the best checkpoint
    load_best_model_at_end=True,

    metric_for_best_model="f1",

    #Logging

    # write log in every 100 step
    logging_steps=100,
    # for WandB , if not write none
    report_to="none",

    # FP16 , use half of memory in 2x gpu
    fp16=torch.cuda.is_available(),


)

# train

In [43]:
train_ds = tokenize_dataset["train"].shuffle(seed=42)
eval_ds  = tokenize_dataset["test"].shuffle(seed=42)

trainer = Trainer(
    model = model,
    args= train_args,
    train_dataset= train_ds,
    eval_dataset= eval_ds,

    #tokenizer = tokenizer
    processing_class = tokenizer,
    data_collator= data_collator,
    compute_metrics= compute_metric,
)

# start training
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.055170,0.274756,0.936053,0.936034
2,0.091780,0.226911,0.941842,0.941818
3,0.050494,0.265311,0.942763,0.942805


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=11250, training_loss=0.07195480283101399, metrics={'train_runtime': 1097.5869, 'train_samples_per_second': 327.992, 'train_steps_per_second': 10.25, 'total_flos': 9734747105491968.0, 'train_loss': 0.07195480283101399, 'epoch': 3.0})

# final eval and  save model

In [44]:
results = trainer.evaluate()
print(f"Accuracy: {results['eval_accuracy']}")
print(f"F1 Score: {results['eval_f1']}")

# save model for use in Gradio
trainer.save_model("./final_model")
tokenizer.save_pretrained("./final_model")

Accuracy: 0.9427631578947369
F1 Score: 0.9428048020485733


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./final_model/tokenizer_config.json', './final_model/tokenizer.json')

In [ ]:
ششششششششششش